# Data Ingestion

## Import Modules

In [1]:
# Make sure you are the parent directory
from pathlib import Path
import sys

# Define directory path: notebooks/ -> project root
PROJECT_ROOT = Path.cwd().resolve().parent
display(PROJECT_ROOT)

# Add it to sys.path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PosixPath('/expanse/lustre/projects/uci157/bguo3/DSC_288R')

In [ ]:
# PySpark libs
from pyspark.sql import functions as F
from pyspark.sql.types import (
    LongType, IntegerType,
    FloatType, StringType, BooleanType
)

# Other libs
import kagglehub
from src.utils.pyspark_helper import create_spark_session, memory_count

## Set up for Spark

In [7]:
# Set up for Spark app & resource allocation
spark = create_spark_session("steam_reviews_data_ingestion")

## Data Ingestion

In [8]:
# Fetch data through Kaggle API
USE_CACHED = True
BASE_ROOT = "/expanse/lustre/scratch/bguo3/temp_project/steam_reviews/raw_csv"
BASE_PATH = kagglehub.dataset_download(
    "kieranpoc/steam-reviews",
    output_dir=BASE_ROOT,
    force_download=False if USE_CACHED else True,
)
print("Downloaded to:", BASE_PATH)

Downloaded to: /expanse/lustre/scratch/bguo3/temp_project/steam_reviews/raw_csv


In [9]:
# Define Spark DF schema
schema = {
    # User/Game Features
    "author_steamid": LongType(),
    "appid": IntegerType(),
    "author_num_games_owned": IntegerType(),
    "author_num_reviews": IntegerType(),
    "author_playtime_forever": IntegerType(),
    "author_playtime_last_two_weeks": IntegerType(),
    "author_playtime_at_review": IntegerType(),
    "author_last_played": LongType(),

    # Review Features
    "review": StringType(),
    "voted_up": BooleanType(),
    "votes_up": IntegerType(),
    "votes_funny": LongType(),
    "weighted_vote_score": FloatType(),
    "comment_count": IntegerType(),
    "written_during_early_access": BooleanType(),
    "language": StringType(),

    # Time Features
    "timestamp_created": LongType(),
    "timestamp_updated": LongType(),
}

cols = list(schema.keys())

In [10]:
# Load Spark DF through CSV
CSV_OPTIONS = {
    "header": "true",
    "inferSchema": "false"
}
# ## Below is the PATH to smaller dataset
# CSV_PATH = f'file:{BASE_PATH}/weighted_score_above_08.csv'
# Below is the PATH to 40GB dataset
CSV_PATH = f'file:{BASE_PATH}/all_reviews/all_reviews.csv'

df = (
    spark.read
        .options(**CSV_OPTIONS)
        .csv(CSV_PATH)
        .select([
            F.col(c).cast(c_type).alias(c)
            for c, c_type in schema.items()
        ])
)

## Display General Info

In [11]:
# Verify to have a correct schema
df.printSchema()

root
 |-- author_steamid: long (nullable = true)
 |-- appid: integer (nullable = true)
 |-- author_num_games_owned: integer (nullable = true)
 |-- author_num_reviews: integer (nullable = true)
 |-- author_playtime_forever: integer (nullable = true)
 |-- author_playtime_last_two_weeks: integer (nullable = true)
 |-- author_playtime_at_review: integer (nullable = true)
 |-- author_last_played: long (nullable = true)
 |-- review: string (nullable = true)
 |-- voted_up: boolean (nullable = true)
 |-- votes_up: integer (nullable = true)
 |-- votes_funny: long (nullable = true)
 |-- weighted_vote_score: float (nullable = true)
 |-- comment_count: integer (nullable = true)
 |-- written_during_early_access: boolean (nullable = true)
 |-- language: string (nullable = true)
 |-- timestamp_created: long (nullable = true)
 |-- timestamp_updated: long (nullable = true)



In [ ]:
# Estimate the size of full dataset after columns removed
memory_count(df)

Total row counts: 113885601 rows
Total estimated size: 54.16 GB


## Write to Parquet File

In [13]:
# Define path
EXPANSE_ROOT = "/expanse/lustre/scratch/bguo3/temp_project"
PARQUET_PATH = f"file:{EXPANSE_ROOT}/steam_reviews/full_parquet"

(
    df.write
        .mode("overwrite")
        .option("compression", "snappy")
        .parquet(PARQUET_PATH)
)
print(f"Saved full parquet to: {PARQUET_PATH}")

Saved full parquet to: file:/expanse/lustre/scratch/bguo3/temp_project/steam_reviews/full_parquet


In [14]:
### END ###

In [ ]:
spark.stop()